## ML - Classifica Sentimento para Label

In [5]:
import collections 
import pandas as pd
import numpy as np
import nltk

df_balance = pd.read_csv('df_balance.csv', encoding = 'utf-8', sep = ';')
df_novo = pd.read_csv('df_novo.csv', encoding = 'utf-8', sep = ';')

classificacao = df_balance["SENTIMENT"].replace(["NEGATIVO","NEUTRO","POSITIVO"], [-1,0,1])

df_balance["CLASSFICATION"] = classificacao

df_balance['COMMENT_TEXT'] = df_balance['COMMENT_TEXT'].str.replace('[^\w\s]','')

C:\ProgramData\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3071: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,
<ipython-input-5-2127df87aed9>:13: FutureWarning: The default value of regex will change from True to False in a future version.
  df_balance['COMMENT_TEXT'] = df_balance['COMMENT_TEXT'].str.replace('[^\w\s]','')


## Tokenização Treino e Testeções

In [6]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import re
import emoji
import unicodedata
import string

####################################
############## Funcoes #############

def preprocess_text(text, remove_stop = True, 
                    stem_words = False, remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bud que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)

def tokenizacao(df):
    
    # cria a coluna com textos vetorizados
    rows, cols = df.shape

    df['TOKEN'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

###################################################
############## Treina o modelo ####################
  
#carregando DataFrame
# df = inputs[0]

#carregando a lista de de Stopwords 
portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

df = tokenizacao(df_balance)


## Vetorização Treino e Teste

In [7]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import  SGDClassifier
from sklearn.svm import SVC

####################################
############## Funcoes #############
  
def treinar_vetorizacao(df):

    # coleciona as palavras usadas para o treinamento da função CountVectorizer
    lista_treino = []

    for item in df['TOKEN']:
        lista_treino1 = [n for n in item if n not in lista_treino]
        lista_treino.extend(lista_treino1)

    # treina o modelo de vetorização
    vectorize = CountVectorizer(lowercase=True, strip_accents='unicode')

    vectorize.fit(lista_treino)
    
    return vectorize, lista_treino

def vectorize2(lista):
    lista2 = [vectorize.vocabulary_[item] for item in lista]
    
    return lista2

###################################################
############## Treina o modelo ####################
  
#carregando DataFrame
#df = inputs[0]
#
vectorize, lista_treino = treinar_vetorizacao(df)

# criando a coluna com o texto vetorizado
df['VECTORS'] = df['TOKEN'].apply(vectorize2)


## Tokenização Novas Entradas

In [8]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import re
import emoji
import unicodedata
import string

####################################
############## Funcoes #############

def preprocess_text(text, remove_stop = True, 
                    stem_words = False, remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bud que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)

def tokenizacao(df):
    
    # cria a coluna com textos vetorizados
    rows, cols = df.shape

    df['TOKEN'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

###################################################
############## Treina o modelo ####################
  
#carregando DataFrame
# df = inputs[0]

#carregando a lista de de Stopwords 
portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

df = tokenizacao(df)


## Treino, Teste e Predição

In [9]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import  SGDClassifier
from sklearn.svm import SVC

####################################
############## Funcoes #############

# função para retornar apenas as palavras da frase, sem a classificação (sentimento)
def busca_Palavras(frases):
    todas_Palavras =[]
    for (palavras, sentimento) in frases:
        todas_Palavras.extend(palavras)
    return todas_Palavras
  
def busca_Palavras2(frases):
    todas_Palavras =[]
    for (palavras) in frases:
        todas_Palavras.extend(palavras)
    return todas_Palavras

# função para verificar a quantidade de vezes que a palavra é mencionada
def busca_frequencia(palavras):
    palavras = nltk.FreqDist(palavras)
    return palavras

# função para retornar somente as palavras únicas
def busca_palavras_unicas(frequencia):
    freq = frequencia.keys()
    return freq

# função para identificar quais palavras únicas estão no documento passado
def extrator_palavras(documento):
    doc = set(documento)
    caracteristicas = {}
    for palavras in palavras_unicas_treino:
        caracteristicas['%s' % palavras] = (palavras in doc)
    return caracteristicas

def extrator_palavras_teste(documento):
    doc = set(documento)
    caracteristicas = {}
    for palavras in palavras_unicas_teste:
        caracteristicas['%s' % palavras] = (palavras in doc)
    return caracteristicas

###################################################
############## Treina o modelo ####################
  
# proporção da base de treino
size_treino=0.3

#carregando DataFrame
# df = inputs[0]
df = df[["TOKEN","CLASSFICATION"]]

treino, teste = train_test_split(df, test_size=size_treino)

treino = [tuple(x) for x in treino.values]
palavras_treino = busca_Palavras(treino)

teste = [tuple(x) for x in teste.values]
palavras_teste = busca_Palavras(teste)

frequencia_treino = busca_frequencia(palavras_treino)
#df_treino = pd.DataFrame(frequencia_treino.most_common(20))
#frequencia_treino.plot(30, cumulative = False)

frequencia_teste = busca_frequencia(palavras_teste)

palavras_unicas_treino = busca_palavras_unicas(frequencia_treino)
palavras_unicas_teste = busca_palavras_unicas(frequencia_teste)

base_completa_treino = nltk.classify.apply_features(extrator_palavras, treino)
base_completa_teste = nltk.classify.apply_features(extrator_palavras_teste, teste)

classificador = nltk.NaiveBayesClassifier.train(base_completa_treino)
tst = nltk.classify.accuracy(classificador, base_completa_teste)

################

def classifica_txt(df1):
  teste = df1[["TOKEN"]]
  comStem = [p for p in teste.values]
  testeStemming.append(str(comStem[0]))
  test = []
  probabilidade = 0
  novo = extrator_palavras(testeStemming)
  distribuicao = classificador.prob_classify(novo)
  result = 0
    
  for classe in distribuicao.samples():
    test.append((str(classe) + " : " + str(distribuicao.prob(classe))))
    if distribuicao.prob(classe) >= probabilidade:
      result = str(classe)
      data = {'Result': [(classe)]}
      probabilidade = distribuicao.prob(classe)
  
  return result

def classifica_df(df):
  # cria a coluna com textos vetorizados
  rows, cols = df.shape
  df['LABEL_PREDICTION'] = [classifica_txt(df["TOKEN"][row]) for row in range(rows)]
  return df


#############

df1 = df #inputs[1]
teste = df1[["TOKEN"]]

#stemmer = nltk.stem.SnowballStemmer('portuguese')
#stemmer = nltk.stem.RSLPStemmer()
'''
testeStemming = []
comStem = [p for p in teste.values]
testeStemming.append(str(comStem[0]))
test = []
probabilidade = 0
novo = extrator_palavras(testeStemming)
distribuicao = classificador.prob_classify(novo)
for classe in distribuicao.samples():
  test.append((str(classe) + " : " + str(distribuicao.prob(classe))))
  if distribuicao.prob(classe) >= probabilidade:
	data = {'Result': [str(classe)]}
	probabilidade = distribuicao.prob(classe) 
'''
df1['SENTIMENT_PREDICTION'] = 'Nan'

for index, row in df1.iterrows():
  testr = row["TOKEN"]
  testr = str(' '.join(testr))
  teste = 'ruim pessimo'
  testeStemming = []
  comStem = ''
  for(palavras_treino) in testr.split(): 
  	comStem = [p for p in palavras_treino.split()]
  	testeStemming.append(str(comStem[0]))
  test = []
  probabilidade = 0
  novo = extrator_palavras(testeStemming)
  distribuicao = classificador.prob_classify(novo)
  result = ''
  for classe in distribuicao.samples():
    if distribuicao.prob(classe) >= probabilidade:
      probabilidade = distribuicao.prob(classe)
      result = str(classe).replace("-1", "Negative").replace("0", "Neutral").replace("1", "Positive")
      data = {'Result': [str(classe)]}
      df1.loc[index,'SENTIMENT_PREDICTION'] = result
      #test.append((str(classe) + " : " + str(distribuicao.prob(classe))))
    #if distribuicao.prob(classe) >= probabilidade:
	#  result = str(classe)
	#data = {'Result': [str(classe)]}
	#  probabilidade = distribuicao.prob(classe)
  #df1.loc[index,'LABEL_PREDICTION'] = result
	
# criando a coluna com o texto vetorizado
#df1['LABEL_PREDICTION'] = df1['TOKEN'].apply(classifica)
#df1 = classifica_df(df1)

#df1['LABEL_PREDICTION'] = df1['TOKEN'].apply(classifica_txt)
#df1['LABEL_PREDICTION'] = [classifica_txt(df1["TOKEN"][row]) for row in range(rows)]
  
  #data = {'Result': ['Probabilidade de ser negativa']}
	#test.append(str(class, distribuicao.prob(classe)))
	
'''	print('%s: %f' % (classe, distribuicao.prob(classe)))
    if classe == -1:
        data = {'Result': ['Probabilidade de ser negativa']}
	elif classe == 0:
    	data = {'Result': ['Probabilidade de ser Neutra']}
    else:
        data = {'Result': ['Probabilidade de ser Positiva']}
'''
#df = pd.DataFrame(data)

<ipython-input-9-b563f0bde717>:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['SENTIMENT_PREDICTION'] = 'Nan'
C:\ProgramData\Anaconda3\lib\site-packages\pandas\core\indexing.py:1720: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


"\tprint('%s: %f' % (classe, distribuicao.prob(classe)))\n    if classe == -1:\n        data = {'Result': ['Probabilidade de ser negativa']}\n\telif classe == 0:\n    \tdata = {'Result': ['Probabilidade de ser Neutra']}\n    else:\n        data = {'Result': ['Probabilidade de ser Positiva']}\n"